# 05B — Feature Extraction (Sliding Window Reference)

This notebook reproduces the `_forked/fb_code/feature_extraction.py` pipeline and CSV layout, modified to use sliding windows with a shift of 1/3 of the window length.

**Compatibility behavior retained:**

- Loads raw `WESAD/S{id}/S{id}.pkl` files.
- Applies filtering, EDA SCR/SCL decomposition, timestamp outer joins, label backfilling, and 60-second sliding windows (shift = 20 seconds / 1/3 window length).
- Keeps the original three WESAD classes: baseline (`1`), stress (`2`), and amusement (`3`).
- Writes wide format with an index column and one-hot target columns `0`, `1`, and `2`.

Output: `Data/features_b/S{id}_feats.csv`

In [2]:
import gc
import pickle
import sys
import time
import warnings
from pathlib import Path
from unittest.mock import MagicMock

import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import scipy.signal as scisig
import scipy.stats
import heartpy as hp
import neurokit2 as nk

warnings.simplefilter('ignore')

# Prevent biosppy's optional interactive plotting modules from requiring tkinter.
sys.modules['biosppy.inter_plotting'] = MagicMock()
sys.modules['biosppy.inter_plotting.ecg'] = MagicMock()
sys.modules['biosppy.inter_plotting.acc'] = MagicMock()
import biosppy

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'WESAD').exists())
FB_CODE_DIR = REPO_ROOT / '_forked' / 'fb_code'
sys.path.insert(0, str(FB_CODE_DIR))

from utils import (
    butter_lowpass_filter,
    filterSignalFIR as filter_signal_fir,
    get_net_accel,
    get_window_stats,
    get_peak_freq,
    get_absolute_integral,
    get_slope,
    get_dynamic_range,
)
from respiration import get_resp_features

WESAD_DIR = REPO_ROOT / 'WESAD'
FEATURE_OUTPUT_DIR = REPO_ROOT / 'Data' / 'features_b'
FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
WINDOW_IN_SECONDS = 60
FS = {'ACC': 32, 'BVP': 64, 'EDA': 4, 'TEMP': 4, 'label': 700,
      'Resp': 700, 'ECG': 700, 'chest': 700}

## 1. Fork-Compatible Helpers

In [3]:
def get_window_stats_ecg(data, label=-1):
    if 'ECG' not in data or data['ECG'].dropna().empty:
        return {
            'bpm': None, 'ibi': None, 'sdnn': None,
            'sdsd': None, 'rmssd': None,
            'pnn20': None, 'pnn50': None,
        }
    try:
        _, measures = hp.process(data['ECG'].dropna().reset_index(drop=True), 700)
        return {
            'bpm': measures['bpm'], 'ibi': measures['ibi'], 'sdnn': measures['sdnn'],
            'sdsd': measures['sdsd'], 'rmssd': measures['rmssd'],
            'pnn20': measures['pnn20'], 'pnn50': measures['pnn50'],
        }
    except Exception:
        return {
            'bpm': None, 'ibi': None, 'sdnn': None,
            'sdsd': None, 'rmssd': None,
            'pnn20': None, 'pnn50': None,
        }

## 2. Fork-Compatible Signal Alignment

The following deliberately reproduces the original timestamp joins and backfilled labels. This is not the recommended pipeline for new modeling work; it exists solely to make results directly comparable with `_forked`.

In [4]:
def load_raw_subject(subject_id):
    path = WESAD_DIR / f'S{subject_id}' / f'S{subject_id}.pkl'
    with open(path, 'rb') as file:
        return pickle.load(file, encoding='latin1')


def compute_features_forked(raw_data, norm_type=None):
    wrist = raw_data['signal']['wrist']
    chest = raw_data['signal']['chest']
    labels = raw_data['label']

    eda_df = pd.DataFrame(wrist['EDA'], columns=['EDA'])
    bvp_df = pd.DataFrame(wrist['BVP'], columns=['BVP'])
    acc_df = pd.DataFrame(wrist['ACC'], columns=['ACC_x', 'ACC_y', 'ACC_z'])
    temp_df = pd.DataFrame(wrist['TEMP'], columns=['TEMP'])
    label_df = pd.DataFrame(labels, columns=['label'])
    acc_c_df = pd.DataFrame(chest['ACC'], columns=['ACC_x_C', 'ACC_y_C', 'ACC_z_C'])
    ecg_df = pd.DataFrame(chest['ECG'], columns=['ECG'])
    eda_c_df = pd.DataFrame(chest['EDA'], columns=['EDA_C'])
    emg_c_df = pd.DataFrame(chest['EMG'], columns=['EMG_C'])
    resp_c_df = pd.DataFrame(chest['Resp'], columns=['Resp_C'])
    temp_c_df = pd.DataFrame(chest['Temp'], columns=['Temp_C'])

    # Exact filter and EDA-decomposition sequence from _forked.
    eda_df['EDA'] = butter_lowpass_filter(eda_df['EDA'], 1.0, FS['EDA'], 6)
    eda_c_df['EDA_C'] = butter_lowpass_filter(eda_c_df['EDA_C'], 1.0, FS['chest'], 6)
    eda_data = nk.eda_phasic(nk.standardize(eda_df['EDA']), sampling_rate=FS['EDA'])
    eda_df['EDA_SCR'] = eda_data['EDA_Phasic']
    eda_df['EDA_SCL'] = eda_data['EDA_Tonic']
    eda_data_c = nk.eda_phasic(nk.standardize(eda_c_df['EDA_C']), sampling_rate=FS['chest'])
    eda_c_df['EDA_SCR_C'] = eda_data_c['EDA_Phasic']
    eda_c_df['EDA_SCL_C'] = eda_data_c['EDA_Tonic']

    for column in acc_df.columns:
        acc_df[column] = filter_signal_fir(acc_df[column].values)
    for column in acc_c_df.columns:
        acc_c_df[column] = filter_signal_fir(acc_c_df[column].values)

    # Memory-efficient alignment to 700Hz master index (replaces heavy outer joins)
    n_700 = len(labels)
    df = pd.DataFrame(index=np.arange(n_700))
    df['label'] = np.asarray(labels).ravel()

    def add_signal(name, arr, fs, val_cols=None):
        arr = np.asarray(arr)
        idx = np.round(np.arange(len(arr)) * 700 / fs).astype(int)
        idx = np.clip(idx, 0, n_700 - 1)
        if arr.ndim == 1:
            s = pd.Series(arr, index=idx).groupby(level=0).mean()
            df[name] = s
        else:
            for c_i, col_name in enumerate(val_cols):
                s = pd.Series(arr[:, c_i], index=idx).groupby(level=0).mean()
                df[col_name] = s

    add_signal('EDA', eda_df['EDA'], FS['EDA'])
    add_signal('EDA_SCR', eda_df['EDA_SCR'], FS['EDA'])
    add_signal('EDA_SCL', eda_df['EDA_SCL'], FS['EDA'])
    add_signal('BVP', bvp_df['BVP'], FS['BVP'])
    add_signal('TEMP', temp_df['TEMP'], FS['TEMP'])
    add_signal('ACC', acc_df.values, FS['ACC'], ['ACC_x', 'ACC_y', 'ACC_z'])
    add_signal('ECG', ecg_df['ECG'], FS['chest'])
    add_signal('EDA_C', eda_c_df['EDA_C'], FS['chest'])
    add_signal('EDA_SCR_C', eda_c_df['EDA_SCR_C'], FS['chest'])
    add_signal('EDA_SCL_C', eda_c_df['EDA_SCL_C'], FS['chest'])
    add_signal('ACC_C', acc_c_df.values, FS['chest'], ['ACC_x_C', 'ACC_y_C', 'ACC_z_C'])
    add_signal('EMG_C', emg_c_df['EMG_C'], FS['chest'])
    add_signal('Resp_C', resp_c_df['Resp_C'], FS['chest'])
    add_signal('Temp_C', temp_c_df['Temp_C'], FS['chest'])

    df['label'] = df['label'].bfill()
    df.reset_index(drop=True, inplace=True)

    if norm_type == 'std':
        df = (df - df.mean()) / df.std()
    elif norm_type == 'minmax':
        df = (df - df.min()) / (df.max() - df.min())

    grouped = df.groupby('label')
    return grouped, grouped.get_group(1), grouped.get_group(2), grouped.get_group(3)

## 3. Fork-Compatible Wide Feature Rows

In [5]:
def get_samples_forked(data, label):
    samples = []
    window_len = FS['label'] * WINDOW_IN_SECONDS
    step = window_len // 3
    feat_names = None

    for start in range(0, len(data) - window_len + 1, step):
        window = data.iloc[start:start + window_len].copy()

        net_acc = get_net_accel(window, 'wrist').rename('net_acc')
        window = pd.concat([net_acc, window], axis=1)
        net_acc_c = get_net_accel(window, 'chest').rename('net_acc_C')
        window = pd.concat([net_acc_c, window], axis=1)

        stats = get_window_stats(window, label=label)
        ecg_stats = get_window_stats_ecg(window, label=label)
        if 'label' in stats:
            del stats['label']
        x = pd.DataFrame(stats)
        if 'label' in x.columns:
            x.drop('label', axis=1, inplace=True)
        if 'label' in x.index:
            x.drop('label', axis=0, inplace=True)

        if feat_names is None:
            feat_names = []
            for row in x.index:
                for col in x.columns:
                    feat_names.append(f'{row}_{col}')

        wdf = pd.DataFrame(x.values.flatten()).T
        wdf.columns = feat_names
        wdf['label'] = label

        wdf['BVP_peak_freq'] = get_peak_freq(window['BVP'].dropna())
        wdf['net_acc_abs_integral'] = get_absolute_integral(window['net_acc'].dropna())
        wdf['ACC_x_abs_integral'] = get_absolute_integral(window['ACC_x'].dropna())
        wdf['ACC_y_abs_integral'] = get_absolute_integral(window['ACC_y'].dropna())
        wdf['ACC_z_abs_integral'] = get_absolute_integral(window['ACC_z'].dropna())
        wdf['net_acc_C_abs_integral'] = get_absolute_integral(window['net_acc_C'].dropna())
        wdf['ACC_x_C_abs_integral'] = get_absolute_integral(window['ACC_x_C'].dropna())
        wdf['ACC_y_C_abs_integral'] = get_absolute_integral(window['ACC_y_C'].dropna())
        wdf['ACC_z_C_abs_integral'] = get_absolute_integral(window['ACC_z_C'].dropna())
        wdf['ACC_x_peak_freq'] = get_peak_freq(window['ACC_x'].dropna())
        wdf['ACC_y_peak_freq'] = get_peak_freq(window['ACC_y'].dropna())
        wdf['ACC_z_peak_freq'] = get_peak_freq(window['ACC_z'].dropna())
        wdf['ACC_x_C_peak_freq'] = None if window['ACC_x_C'].dropna().empty else get_peak_freq(window['ACC_x_C'].dropna())
        wdf['ACC_y_C_peak_freq'] = None if window['ACC_y_C'].dropna().empty else get_peak_freq(window['ACC_y_C'].dropna())
        wdf['ACC_z_C_peak_freq'] = None if window['ACC_z_C'].dropna().empty else get_peak_freq(window['ACC_z_C'].dropna())
        for key, value in ecg_stats.items():
            wdf[f'ECG_{key}'] = value
        wdf['EDA_slope'] = get_slope(window['EDA'].dropna())
        wdf['EDA_C_slope'] = None if window['ACC_z_C'].dropna().empty else get_slope(window['EDA_C'].dropna())
        wdf['EDA_drange'] = get_dynamic_range(window['EDA'].dropna())
        wdf['EDA_C_drange'] = None if window['EDA_C'].dropna().empty else get_dynamic_range(window['EDA_C'].dropna())
        wdf['EMG_drange'] = get_dynamic_range(window['EMG_C'].dropna())
        wdf['EMG_abs_integral'] = get_absolute_integral(window['EMG_C'].dropna())
        if not window['Resp_C'].dropna().empty:
            resp_features = get_resp_features(window['Resp_C'].dropna())
            for r_name, r_val in zip([
                'Resp_C_rate', 'Resp_C_Inhal_mean', 'Resp_C_Inhal_std',
                'Resp_C_Exhal_mean', 'Resp_C_Exhal_std', 'Resp_C_I/E',
            ], resp_features):
                wdf[r_name] = r_val
        wdf['TEMP_drange'] = get_dynamic_range(window['TEMP'].dropna())
        wdf['TEMP_C_drange'] = None if window['Temp_C'].dropna().empty else get_dynamic_range(window['Temp_C'].dropna())
        wdf['TEMP_slope'] = get_slope(window['TEMP'].dropna())
        wdf['TEMP_C_slope'] = None if window['Temp_C'].dropna().empty else get_slope(window['Temp_C'].dropna())
        samples.append(wdf)

    if not samples:
        return pd.DataFrame()
    return pd.concat(samples, ignore_index=True)


def extract_subject_forked(subject_id):
    raw_data = load_raw_subject(subject_id)
    _, baseline, stress, amusement = compute_features_forked(raw_data, norm_type=None)

    # Sliding window with step = 1/3 of window length (20 seconds for 60s window)
    baseline_samples = get_samples_forked(baseline, label=1)
    stress_samples = get_samples_forked(stress, label=2)
    amusement_samples = get_samples_forked(amusement, label=0)
    features = pd.concat([baseline_samples, stress_samples, amusement_samples], ignore_index=True)

    features = pd.concat([features.drop(columns=['label']), pd.get_dummies(features['label']).rename(columns=str)], axis=1)

    return features

## 4. Single-Subject Walkthrough

In [6]:
subject_id = SUBJECTS[0]
forked_features_df = extract_subject_forked(subject_id)
print(f'Extracted {len(forked_features_df)} fork-compatible rows for S{subject_id}')
label_cols = [c for c in forked_features_df.columns if str(c) in ['0', '1', '2']]
print(f'{len(forked_features_df.columns)} columns; label counts:\n{forked_features_df[label_cols].sum().to_dict()}')
forked_features_df.head()

KeyboardInterrupt: 

## 5. Process and Save All Subjects

In [ ]:
SKIP_EXTRACTED_SUBJECT = False
summary = []

for subject_id in SUBJECTS:
    output_path = FEATURE_OUTPUT_DIR / f'S{subject_id}_feats.csv'
    if SKIP_EXTRACTED_SUBJECT and output_path.exists():
        print(f'S{subject_id}: already extracted -> skip')
        continue

    started_at = time.time()
    forked_features_df = extract_subject_forked(subject_id)

    # index=True is deliberate: _forked's DataFrame.to_csv used its default index=True.
    forked_features_df.to_csv(output_path, index=True)
    summary.append({
        'subject': subject_id,
        'n_rows': len(forked_features_df),
        'n_features': len(forked_features_df.columns),
        'saved_to': output_path.name,
        'seconds': round(time.time() - started_at, 2),
    })
    print(summary[-1])
    del forked_features_df
    gc.collect()

pd.DataFrame(summary)

{'subject': 2, 'n_rows': 99, 'n_features': 121, 'saved_to': 'S2_feats.csv', 'seconds': 30.41}
{'subject': 3, 'n_rows': 101, 'n_features': 121, 'saved_to': 'S3_feats.csv', 'seconds': 30.6}
{'subject': 4, 'n_rows': 100, 'n_features': 121, 'saved_to': 'S4_feats.csv', 'seconds': 33.22}
{'subject': 5, 'n_rows': 103, 'n_features': 121, 'saved_to': 'S5_feats.csv', 'seconds': 29.62}
{'subject': 6, 'n_rows': 103, 'n_features': 121, 'saved_to': 'S6_feats.csv', 'seconds': 30.88}
{'subject': 7, 'n_rows': 103, 'n_features': 121, 'saved_to': 'S7_feats.csv', 'seconds': 29.5}
{'subject': 8, 'n_rows': 103, 'n_features': 121, 'saved_to': 'S8_feats.csv', 'seconds': 29.37}
{'subject': 9, 'n_rows': 103, 'n_features': 121, 'saved_to': 'S9_feats.csv', 'seconds': 29.14}
{'subject': 10, 'n_rows': 107, 'n_features': 121, 'saved_to': 'S10_feats.csv', 'seconds': 67.96}
{'subject': 11, 'n_rows': 105, 'n_features': 121, 'saved_to': 'S11_feats.csv', 'seconds': 78.41}
{'subject': 13, 'n_rows': 105, 'n_features': 121,

,subject,n_rows,n_features,saved_to,seconds
0,2,99,121,S2_feats.csv,30.41
1,3,101,121,S3_feats.csv,30.60
2,4,100,121,S4_feats.csv,33.22
3,5,103,121,S5_feats.csv,29.62
4,6,103,121,S6_feats.csv,30.88
5,7,103,121,S7_feats.csv,29.50
6,8,103,121,S8_feats.csv,29.37
7,9,103,121,S9_feats.csv,29.14
8,10,107,121,S10_feats.csv,67.96
9,11,105,121,S11_feats.csv,78.41


## 6. Inspect a Fork-Compatible CSV

In [ ]:
sample_file = next(FEATURE_OUTPUT_DIR.glob('S*_feats.csv'), None)
if sample_file is None:
    raise FileNotFoundError('No fork-compatible feature files found. Run Section 5 first.')

forked_features_df = pd.read_csv(sample_file, index_col=0)
print(f'Inspecting: {sample_file.name} ({len(forked_features_df)} rows, {len(forked_features_df.columns)} columns)')
print('Label counts:')
label_cols = [c for c in forked_features_df.columns if str(c) in ['0', '1', '2']]
display(forked_features_df[label_cols].sum().rename('n_windows').to_frame())
forked_features_df.head()

Inspecting: S10_feats.csv (107 rows, 121 columns)
Label counts:


,n_windows
0,16
1,57
2,34


,net_acc_C_mean,net_acc_C_std,net_acc_C_min,net_acc_C_max,net_acc_mean,net_acc_std,net_acc_min,net_acc_max,EDA_mean,EDA_std,...,Resp_C_Exhal_mean,Resp_C_Exhal_std,Resp_C_I/E,TEMP_drange,TEMP_C_drange,TEMP_slope,TEMP_C_slope,0,1,2
0,0.912609,0.006683,0.852244,1.017326,61.898628,2.494550,47.503285,63.264585,0.397236,0.038724,...,1.896476,0.230302,1.026716,1.003310,1.008774,0.000288,0.000003,0,1,0
1,0.914854,0.007196,0.852244,1.017326,62.376334,2.027992,47.503285,63.164460,0.395096,0.036828,...,1.847238,0.207785,1.089503,1.001502,1.009849,0.000059,0.000003,0,1,0
2,0.916966,0.006994,0.852244,1.017326,62.632851,1.561210,48.991811,63.164460,0.378741,0.003812,...,1.822952,0.232349,1.095972,1.000901,1.010341,0.000040,0.000004,0,1,0
3,0.918661,0.001884,0.907689,0.925989,62.652162,1.518335,48.991811,63.164460,0.378929,0.003490,...,1.738214,0.226925,1.090662,1.000901,1.010335,-0.000005,0.000004,0,1,0
4,0.918633,0.001923,0.907689,0.926323,62.461431,1.663748,48.991811,63.120734,0.379111,0.003350,...,1.672222,0.133400,0.950451,1.001502,1.009063,-0.000102,0.000003,0,1,0


## 7. Model Training, Validation, and Serialization (Proposal Methodology)

In [9]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

# Ensure Models directory exists
MODELS_DIR = REPO_ROOT / 'Models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Load all extracted subject feature datasets from Data/features_b/
feature_files = list(FEATURE_OUTPUT_DIR.glob('S*_feats.csv'))
if not feature_files:
    raise FileNotFoundError(f"No feature files found in {FEATURE_OUTPUT_DIR}. Please run feature extraction first.")

all_dfs = []
for f in feature_files:
    df_sub = pd.read_csv(f, index_col=0)
    all_dfs.append(df_sub)

combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"Loaded {len(feature_files)} subjects. Combined dataset shape: {combined_df.shape}")

# Separate features (X) and labels (y)
label_cols = [c for c in combined_df.columns if str(c) in ['0', '1', '2']]
if label_cols and len(label_cols) == 3:
    y = combined_df[label_cols].idxmax(axis=1).astype(int)
    X = combined_df.drop(columns=label_cols)
elif 'label' in combined_df.columns:
    y = combined_df['label'].astype(int)
    X = combined_df.drop(columns=['label'])
else:
    y = combined_df.iloc[:, -1].astype(int)
    X = combined_df.iloc[:, :-1]

# Handle NaN values in features
X = X.fillna(0)

# Train-test split based on proposal (80% train, 20% test, stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}, Testing set: {X_test.shape}")

# Define the 4 classification algorithms specified in the Proposal
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'XGBoost': XGBClassifier(eval_metric='mlogloss', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'accuracy': acc,
        'confusion_matrix': cm,
        'classification_report': report
    }
    
    print(f"{name} Accuracy: {acc:.4f}")
    print("Classification Report:\n", report)
    
    # Save model artifact
    model_filename = MODELS_DIR / f"{name.lower().replace(' ', '_')}_model.joblib"
    joblib.dump(model, model_filename)
    print(f"Saved {name} model to {model_filename}")

print("\nAll 4 proposal models trained, evaluated, and saved successfully!")

Loaded 15 subjects. Combined dataset shape: (1551, 121)
Training set: (1240, 118), Testing set: (311, 118)

--- Training Random Forest ---
Random Forest Accuracy: 1.0000
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        48
           1       1.00      1.00      1.00       170
           2       1.00      1.00      1.00        93

    accuracy                           1.00       311
   macro avg       1.00      1.00      1.00       311
weighted avg       1.00      1.00      1.00       311

Saved Random Forest model to e:\Rizqy\Kuliah\Skripsi\repo-skripsi\Models\random_forest_model.joblib

--- Training SVM ---
SVM Accuracy: 0.7203
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.38      0.55        48
           1       0.75      0.86      0.80       170
           2       0.61      0.63      0.62        93

    accuracy                           0